In [ ]:
from warnings import filterwarnings
filterwarnings('ignore')
from Universos.indices import EMERGENTES
from yfinance import download
import pandas as pd
import numpy as np
import plotly.express as px
import math
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
emergentes = EMERGENTES

In [327]:
emergentes['^BVSP']['pais']

'Brasil'

In [328]:
tickers = list(emergentes.keys())
tickers

['^BVSP',
 '^HSI',
 '000001.SS',
 '^MXX',
 '^IPSA',
 '^MERV',
 '^KLSE',
 '^NSEI',
 '^SET.BK',
 '^JKSE',
 '^KS11',
 '^STI',
 '^IBEX',
 'XU100.IS',
 'IMOEX.ME',
 'WIG20.WA',
 'J203.JO',
 'EGX30.CA',
 'PSEI.PS']

In [329]:
df_emer = download(tickers, period='10y', interval='1mo', auto_adjust=False)

# Selecionar apenas a coluna 'Close'
df_emer = df_emer['Close'] 

# Renomear as colunas para substituir '.' por '_'
df_emer.columns = df_emer.columns.str.replace('.', '_', regex=False)

# Remover todas as colunas que possuem apenas valores faltantes
df_emer = df_emer.dropna(axis=1, how='all') 

# Remover colunas com mais de 20% de valores faltantes
threshold = 0.2  # limite de 20%
df_emer = df_emer.dropna(axis=1, thresh=int((1-threshold)*len(df_emer)))

# Remover a última linha se todos os valores forem 0
if (df_emer.iloc[-1] == 0).all():
    df_emer = df_emer.iloc[:-1]

# Remove a última linha se qualquer valor for NaN
if df_emer.iloc[-1].isna().any():
    df_emer = df_emer.iloc[:-1]

df_emer_ret = df_emer.pct_change(1).dropna()

# Renomear as colunas para indicar que são retornos
df_emer_ret.columns = [col+'_ret' for col in df_emer_ret.columns]


[**********************53%                       ]  10 of 19 completed$J203.JO: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")
[**********************74%***********            ]  14 of 19 completed$EGX30.CA: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  19 of 19 completed

2 Failed downloads:
['J203.JO', 'EGX30.CA']: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


In [330]:
df_emer.tail(2)

Ticker,000001_SS,XU100_IS,^BVSP,^HSI,^IBEX,^JKSE,^KLSE,^KS11,^MERV,^MXX,^NSEI,^STI
Date,,,,,,,,,,,,
2026-01-01,4117.948242,13838.299805,181364.000000,27387.109375,17880.900391,8329.606445,1740.880005,5224.359863,3199554.0,67598.953125,25320.650391,4905.129883
2026-02-01,4082.072510,14413.480469,186464.296875,26705.939453,18150.900391,8310.226562,1741.260010,5507.009766,2816128.0,71155.687500,25819.349609,4938.580078


In [331]:
df_emer_ret.tail(3)

,000001_SS_ret,XU100_IS_ret,^BVSP_ret,^HSI_ret,^IBEX_ret,^JKSE_ret,^KLSE_ret,^KS11_ret,^MERV_ret,^MXX_ret,^NSEI_ret,^STI_ret
Date,,,,,,,,,,,,
2025-12-01,0.020636,0.033288,0.012906,-0.008831,0.057184,0.016246,0.047143,0.073239,0.008309,0.011188,-0.002799,0.027023
2026-01-01,0.037570,0.228815,0.125611,0.068534,0.033112,-0.036699,0.036170,0.239713,0.048478,0.051170,-0.030959,0.055727
2026-02-01,-0.008712,0.041564,0.028122,-0.024872,0.015100,-0.002327,0.000218,0.054102,-0.119837,0.052615,0.019695,0.006819


In [332]:
# =====================
# REGRESSÃO LINEAR PARA CADA TICKER
# + DISTÂNCIA DA REGRESSÃO
# =====================
x = np.arange(len(df_emer))
for ticker in df_emer.columns:
    y = df_emer[ticker].values
    mask = ~np.isnan(y)
    
    if mask.sum() > 1:
        coef = np.polyfit(x[mask], y[mask], 1)
        reg = coef[0] * x + coef[1]

        # adiciona regressão
        df_emer[f"{ticker}_reg_lin"] = reg

        # adiciona distância da regressão
        df_emer[f"{ticker}_dist_reg_lin"] = y - reg
    else:
        df_emer[f"{ticker}_reg_lin"] = np.nan
        df_emer[f"{ticker}_dist_reg_lin"] = np.nan


In [333]:
df_emer.tail(2)

Ticker,000001_SS,XU100_IS,^BVSP,^HSI,^IBEX,^JKSE,^KLSE,^KS11,^MERV,^MXX,^NSEI,^STI,000001_SS_reg_lin,000001_SS_dist_reg_lin,XU100_IS_reg_lin,XU100_IS_dist_reg_lin,^BVSP_reg_lin,^BVSP_dist_reg_lin,^HSI_reg_lin,^HSI_dist_reg_lin,^IBEX_reg_lin,^IBEX_dist_reg_lin,^JKSE_reg_lin,^JKSE_dist_reg_lin,^KLSE_reg_lin,^KLSE_dist_reg_lin,^KS11_reg_lin,^KS11_dist_reg_lin,^MERV_reg_lin,^MERV_dist_reg_lin,^MXX_reg_lin,^MXX_dist_reg_lin,^NSEI_reg_lin,^NSEI_dist_reg_lin,^STI_reg_lin,^STI_dist_reg_lin
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-01-01,4117.948242,13838.299805,181364.000000,27387.109375,17880.900391,8329.606445,1740.880005,5224.359863,3199554.0,67598.953125,25320.650391,4905.129883,3427.885613,690.062630,9497.182202,4341.117602,145865.138039,35498.861961,20763.944363,6623.165012,11889.355209,5991.545182,7565.189571,764.416875,1496.425986,244.454019,3116.038148,2108.321716,1.674121e+06,1.525433e+06,56062.869385,11536.083740,24743.193462,577.456928,3720.670395,1184.459487
2026-02-01,4082.072510,14413.480469,186464.296875,26705.939453,18150.900391,8310.226562,1741.260010,5507.009766,2816128.0,71155.687500,25819.349609,4938.580078,3431.774875,650.297635,9594.339246,4819.141223,146588.040214,39876.256661,20710.564378,5995.375075,11923.654714,6227.245677,7585.326625,724.899938,1494.642607,246.617403,3125.883307,2381.126459,1.693617e+06,1.122511e+06,56179.625484,14976.062016,24901.468700,917.880910,3728.260210,1210.319868


In [334]:
# =====================
# PEGAR ULTIMO VALOR DE DISTÂNCIA
# =====================

dist_cols = [col for col in df_emer.columns if col.endswith('_dist_reg_lin')]

df_dist = (
    df_emer[dist_cols]
    .iloc[-1]
    .dropna()
)

# remover sufixo
df_dist.index = df_dist.index.str.replace('_dist_reg_lin', '')

# =====================
# NORMALIZAR
# =====================

df_dist_norm = df_dist / df_dist.abs().max()

# =====================
# CRIAR LABELS COMPLETOS
# =====================

labels = [
    f"{ticker} | {emergentes[ticker.replace('_', '.')]['pais']}"
    for ticker in df_dist.index
]

df_plot = df_dist_norm.copy()
df_plot.index = labels

# =====================
# ORDENAR
# =====================

df_plot = df_plot.sort_values(ascending=True)

# =====================
# PLOT
# =====================

fig = px.bar(
    x=df_plot.values,
    y=df_plot.index,
    orientation='h',
    title='Distância Normalizada vs Regressão Linear'
)

fig.update_layout(
    template="plotly_white",
    height=650,
    title_x=0.5,
    showlegend=False
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

fig.show()


In [335]:
ticker = '^MERV'
fig = px.line(
    df_emer,
    x=df_emer.index,
    y=[ticker, f"{ticker}_reg_lin"],
    title=f'{ticker} vs Regressão Linear'
)
fig.show()


In [336]:
# =====================
# PEGAR ÚLTIMA LINHA
# =====================

df_last = df_emer_ret.iloc[-1].dropna()

# remover sufixo
df_last.index = df_last.index.str.replace('_ret', '')

# =====================
# ORDENAR
# =====================

df_last = df_last.sort_values(ascending=True)

# =====================
# PLOT
# =====================

fig = px.bar(
    x=df_last.values,
    y=df_last.index,
    orientation='h',
    title=f'Retorno Mensal - {df_emer_ret.index[-1].strftime("%Y-%m")}'
)

fig.update_layout(
    template="plotly_white",
    height=650,
    title_x=0.5,
    legend_title=None
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

fig.show()


In [349]:
# ==========================
# PARAMETROS
# ==========================

cols = [col for col in df_emer.columns if not '_reg_lin' in col and not '_dist_reg_lin' in col]

num_colunas = 3
num_plots = len(cols)
num_linhas = math.ceil(num_plots / num_colunas)


# ==========================
# SUBPLOTS
# ==========================

fig = make_subplots(
    rows=num_linhas,
    cols=num_colunas,
    subplot_titles=cols,
    vertical_spacing=0.5 / num_linhas,
    horizontal_spacing=0.05
)


# ==========================
# LOOP
# ==========================

for i, col in enumerate(cols):

    linha = i // num_colunas + 1
    coluna = i % num_colunas + 1

    serie = df_emer[col].dropna()

    ultimo_valor = serie.iloc[-1]

    # HISTOGRAMA
    hist = go.Histogram(
        x=serie,
        nbinsx=50,
        name=col,
        showlegend=False,
        opacity=0.75,
        marker_color="#1f77b4"   # cor fixa
    )

    fig.add_trace(
        hist,
        row=linha,
        col=coluna
    )

    # LINHA VERTICAL DO ULTIMO VALOR
    fig.add_vline(
        x=ultimo_valor,
        line_width=3,
        line_dash="dash",
        line_color="red",
        row=linha,
        col=coluna
    )


# ==========================
# LAYOUT
# ==========================

fig.update_layout(
    title="Distribuição dos Preços + Último Valor",
    height=650,
    title_x=0.5,
    template="plotly_white",
    showlegend=False
)


fig.show()


In [355]:
# ==========================
# ESCOLHER ATIVO
# ==========================

ticker = "^BVSP"   # altere aqui

serie = df_emer[ticker].dropna()

ultimo_valor = serie.iloc[-1]

# ==========================
# HISTOGRAMA
# ==========================

fig = px.histogram(
    x=serie,
    nbins=60,
    title=f"Distribuição do Preço - {ticker}"
)

# linha vertical do preço atual
fig.add_vline(
    x=ultimo_valor,
    line_width=3,
    line_dash="dash",
    line_color="red"
)

# layout
fig.update_layout(
    template="plotly_white",
    height=500,
    title_x=0.5,
    showlegend=False
)

fig.show()


In [350]:
# ==========================
# PARAMETROS
# ==========================

cols = [col for col in df_emer_ret.columns if '_ret' in col]

num_colunas = 3
num_plots = len(cols)
num_linhas = math.ceil(num_plots / num_colunas)


# ==========================
# SUBPLOTS
# ==========================

fig = make_subplots(
    rows=num_linhas,
    cols=num_colunas,
    subplot_titles=cols,
    vertical_spacing=0.5 / num_linhas,
    horizontal_spacing=0.05
)


# ==========================
# LOOP
# ==========================

for i, col in enumerate(cols):

    linha = i // num_colunas + 1
    coluna = i % num_colunas + 1

    serie = df_emer_ret[col].dropna()

    ultimo_valor = serie.iloc[-1]

    # HISTOGRAMA
    hist = go.Histogram(
        x=serie,
        nbinsx=50,
        name=col,
        showlegend=False,
        opacity=0.75,
        marker_color="#1f77b4"   # cor fixa
    )

    fig.add_trace(
        hist,
        row=linha,
        col=coluna
    )

    # LINHA VERTICAL DO ULTIMO VALOR
    fig.add_vline(
        x=ultimo_valor,
        line_width=3,
        line_dash="dash",
        line_color="red",
        row=linha,
        col=coluna
    )


# ==========================
# LAYOUT
# ==========================

fig.update_layout(
    title="Distribuição dos Retornos + Último Valor",
    height=650,
    title_x=0.5,
    template="plotly_white",
    showlegend=False
)


fig.show()


In [356]:
# ==========================
# ESCOLHER ATIVO
# ==========================

ticker = "^BVSP"   # altere aqui

serie = df_emer_ret[f'{ticker}_ret'].dropna()

ultimo_valor = serie.iloc[-1]

# ==========================
# HISTOGRAMA
# ==========================

fig = px.histogram(
    x=serie,
    nbins=60,
    title=f"Distribuição do Preço - {ticker}"
)

# linha vertical do preço atual
fig.add_vline(
    x=ultimo_valor,
    line_width=3,
    line_dash="dash",
    line_color="red"
)

# layout
fig.update_layout(
    template="plotly_white",
    height=500,
    title_x=0.5,
    showlegend=False
)

fig.show()
